# <strong> ADM & LT 2024/2025: HACKATON RAG

**Students:**
- Francesco Lazzari - 1917922
- Gabriel Pinos - 1965035

**Note:**

*For some submission we tried with to implement some variation of the following code like HNSW index, fine-tuning the reader or encoder model and different prompt templates. However this should be the code used for the submission that achieved the best results in the leaderboard.*

In [ ]:
!pip install -q torch transformers==4.49 accelerate bitsandbytes langchain langchain-experimental langchain-community sentence-transformers faiss-cpu langchain-huggingface ragatouille

### **LIBRARIES**

In [ ]:
# ==============================
# Standard Library
# ==============================
import os
import re
import gc
import ast
import time
import json
import shutil
import warnings
from typing import List, Dict, Optional, Tuple

# ==============================
# Progress & Logging
# ==============================
from tqdm import tqdm

# ==============================
# Data Handling & Visualization
# ==============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# PyTorch & CUDA
# ==============================
import torch

# ==============================
# HuggingFace Transformers
# ==============================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    Pipeline,
    pipeline
)

# ==============================
# HuggingFace Hub
# ==============================
from huggingface_hub import login

# ==============================
# PEFT (Parameter-Efficient Fine-Tuning)
# ==============================
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    PeftModel
)

# ==============================
# Datasets
# ==============================
import datasets
from datasets import Dataset, DatasetDict

# ==============================
# Langchain & Extensions
# ==============================
from langchain.schema import Document as LangchainDocument
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

# ==============================
# RAGatouille
# ==============================
from ragatouille import RAGPretrainedModel

# ==============================
# Sentence Transformers
# ==============================
from sentence_transformers import SentenceTransformer

# ==============================
# Local Tools
# ==============================
import ace_tools as tools

# ==============================
# Suppress Warnings
# ==============================
warnings.filterwarnings('ignore')


### **DATA LOADING**

In [ ]:
corpus_df = pd.read_csv("input/adm-lt-2024-2025-hackathon-rag/corpus.csv")
train_df = pd.read_csv("input/adm-lt-2024-2025-hackathon-rag/train.csv")

### **EMBEDDING MODEL SELECTION**

We evaluated several embedding models (both general and biomedical-specific) to identify the best one for passage retrieval.  
Each model was tested using a dynamic Precision@k metric based on ground truth passage IDs.  

The best-performing model was **`pritamdeka/S-PubMedBert-MS-MARCO`**, which we selected for the RAG pipeline.


In [ ]:
# List of models to test for biomedical retrieval
models_to_test = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    "pritamdeka/S-PubMedBert-MS-MARCO",
    "sentence-transformers/msmarco-MiniLM-L6-cos-v5",
    "allenai/specter2_base",
    "cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    "gsarti/scibert-nli",
    "sentence-transformers/pubmedbert-base-sentence-transformer"
]

retriever_eval_results = []

In [ ]:
# Convert corpus DataFrame into LangchainDocument format
documents = [
    LangchainDocument(page_content=row["passage"], metadata={"id": int(row["id"])})
    for _, row in corpus_df.iterrows()
]

# Function to evaluate a retriever model
# Each query has a variable number of relevant documents
def evaluate_model(model_name: str) -> Dict:
    print(f"\n- Evaluating model: {model_name}")
    
    # Load the embedding model
    embedding_model = HuggingFaceEmbeddings(model_name=model_name)
    vector_db = FAISS.from_documents(documents=documents, embedding=embedding_model)
    retriever = vector_db.as_retriever()

    matched_questions = 0
    total_matches = 0
    total_questions = len(train_df)

    for _, row in tqdm(train_df.iterrows(), total=total_questions):
        query = row["question"]
        raw = row["relevant_passage_ids"]
        
        # Fix issues where spaces may be used instead of commas
        cleaned = raw.replace(" ", ",")
        try:
            relevant_ids = set(ast.literal_eval(cleaned))
        except:
            relevant_ids = set()

        k_dynamic = len(relevant_ids)  # Use number of relevant documents as dynamic k

        if k_dynamic == 0:
            continue  # Skip queries with no ground truth

        try:
            results = retriever.get_relevant_documents(query=query, search_kwargs={"k": k_dynamic})
        except:
            continue

        retrieved_ids = set()
        for doc in results:
            doc_id = doc.metadata.get("id")
            if doc_id is not None:
                retrieved_ids.add(int(doc_id))

        hits = relevant_ids.intersection(retrieved_ids)
        total_matches += len(hits)
        if hits:
            matched_questions += 1

    precision = matched_questions / total_questions
    avg_hits = total_matches / total_questions

    print(f"- Precision: {precision:.2%}")
    print(f"- Avg Hits: {avg_hits:.2f}")

    return {
        "model": model_name,
        "precision": precision,
        "avg_hits": avg_hits
    }

# Apply evaluation to all defined models
for model in models_to_test:
    result = evaluate_model(model)
    retriever_eval_results.append(result)

# Save evaluation results to a JSON file
with open("retriever_eval_log.json", "w") as f:
    json.dump(retriever_eval_results, f, indent=2)

### **SEMANTIC DOC SPLITTING FUNCTION**

In this section, we split each document into semantically coherent chunks suitable for retrieval.  
We use `SemanticChunker` when possible, and fall back to a sentence-based splitter if needed.  
If the semantic chunker is used for a given document then long or short chunks are filtered manually, and overlapping context is added to improve retrieval quality.

In [ ]:
# ===============================================================
# Support Functions for Document Splitting
# ===============================================================

def count_tokens(text: str) -> int:
    """
    Count the number of tokens using the selected model's tokenizer.

    Returns the number of tokens in the input text using the global tokenizer.
    If tokenization fails, falls back to an approximate token count.
    """
    global TOKENIZER
    try:
        # Tokenize without special tokens for a clean count
        tokens = TOKENIZER.encode(text, add_special_tokens=False, truncation=False)
        return len(tokens)
    except Exception:
        # Fallback in case of tokenizer failure (should not happen)
        return len(text) // 4


def split_if_too_big(text: str, max_tokens: int, min_tokens: int, doc_idx: int, chunk_idx: int) -> List[str]:
    """
    Handles cases where SemanticChunker produces overly long chunks.

    Steps:
    1. Check if the chunk is too small or already valid in size
    2. If too large, split by sentence using regex
    3. Rebuild valid chunks respecting token constraints
    4. Truncate single sentences that exceed the limit
    """
    current_tokens = count_tokens(text)

    # Skip chunks that are too small (likely not useful for RAG)
    if current_tokens < min_tokens:
        return []

    # Acceptable size, return as-is
    if current_tokens <= max_tokens:
        return [text]

    print("=" * 50)
    print("\n")
    print(f"\n Chunk {doc_idx}_{chunk_idx} is too large: {current_tokens} tokens\n")

    # Quick regex-based sentence splitting (faster than nltk/spacy)
    sentences = re.split(r'[.!?]+\s+', text)  # similar to SemanticChunker's regex

    # Clean up whitespace and filter out short sentences
    all_sentences = [s.strip() for s in sentences if len(s.strip()) > 15]

    chunks = []
    current_chunk_sentences = []
    current_tokens = 0

    for sentence in all_sentences:
        sentence_tokens = count_tokens(sentence)

        # Handle individual sentences that are too long
        if sentence_tokens > max_tokens:
            if current_tokens >= min_tokens:
                chunk_text = " ".join(current_chunk_sentences)
                chunks.append(chunk_text)

            # Truncate long sentence by words
            words = sentence.split()
            truncated_words = []
            temp_tokens = 0

            for word in words:
                test_text = " ".join(truncated_words + [word])
                test_tokens = count_tokens(test_text)
                if test_tokens > max_tokens:
                    break
                truncated_words.append(word)
                temp_tokens = test_tokens

            if temp_tokens >= min_tokens:
                truncated_sentence = " ".join(truncated_words)
                chunks.append(truncated_sentence)

            # Reset current chunk
            current_chunk_sentences = []
            current_tokens = 0
            continue

        # Regular logic for valid-length sentences
        if current_tokens + sentence_tokens > max_tokens:
            if current_tokens >= min_tokens:
                chunk_text = " ".join(current_chunk_sentences)
                chunks.append(chunk_text)

            current_chunk_sentences = [sentence]
            current_tokens = sentence_tokens
        else:
            current_chunk_sentences.append(sentence)
            current_tokens += sentence_tokens

    # Final chunk if anything remains
    if current_tokens >= min_tokens:
        chunk_text = " ".join(current_chunk_sentences)
        chunks.append(chunk_text)

    print(f"- Chunk {doc_idx}_{chunk_idx} split into {len(chunks)} valid chunks\n")
    return chunks


def create_overlap(
    chunks: List[str],
    overlap_ratio: float = 0.10,
    max_chunk_tokens: int = 450
) -> List[str]:
    """
    Adds overlap between adjacent chunks while preserving sentence boundaries.

    For each chunk (except the first), append the last N sentences from the previous
    chunk based on the given overlap ratio, provided the combined chunk size remains valid.
    """
    if len(chunks) <= 1 or overlap_ratio <= 0:
        return chunks

    overlapped_chunks = []

    for i, chunk in enumerate(chunks):
        current_chunk = chunk

        if i > 0:
            prev_chunk = chunks[i - 1]
            prev_sentences = re.split(r'[.!?]+\s+', prev_chunk)

            # Determine number of sentences to overlap
            overlap_sentences_count = max(1, int(len(prev_sentences) * overlap_ratio))
            overlap_text = '. '.join(prev_sentences[-overlap_sentences_count:])

            # Combine and check if within token limit
            combined_text = overlap_text + " " + current_chunk
            if count_tokens(combined_text) <= max_chunk_tokens:
                current_chunk = combined_text

        overlapped_chunks.append(current_chunk)

    return overlapped_chunks


In [ ]:
# ===============================================================
# Knowledge Base Chunk Splitting Based on Semantic Similarities
# ===============================================================

def create_doc_chunks(
    documents: List[LangchainDocument],
    model,
    max_chunk_tokens: int = 450,
    min_chunk_tokens: int = 50,
    similarity_threshold: float = 0.76,
    device: Optional[str] = None,
    batch_size: int = 32,
    overlap_ratio: float = 0.10
) -> List[LangchainDocument]:
    """
    Splits a list of Langchain documents into semantically meaningful chunks
    using SemanticChunker and a fallback strategy. Adds overlapping context
    when needed and applies post-filtering and metadata tagging.
    """

    print(f"- Splitting {len(documents)} documents into chunks\n")
    print(f"- Target size: {max_chunk_tokens} tokens per chunk\n")

    # Device configuration
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"- Device: {device}\n")

    # Initialize semantic chunker
    semantic_chunker = None

    try:
        semantic_chunker = SemanticChunker(
            embeddings=model,
            breakpoint_threshold_type="gradient",
            breakpoint_threshold_amount=85,  
            min_chunk_size=min_chunk_tokens,
            buffer_size=2
        )
        print(f"- SemanticChunker ready with threshold {similarity_threshold}\n")
    except Exception as e:
        print(f"- Error setting up SemanticChunker: {e}\n")
        print("- Proceeding with fallback splitter only\n")
        semantic_chunker = None

    # Fallback sentence-based character splitter in case semantic chunking fails
    separators = [
        "\n#{1,6} ",
        "```\n",
        "\n\\*\\*\\*+\n",
        "\n---+\n",
        "\n___+\n",
        "\n\n",
        "\n",
        " ",
        "",
    ]
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_tokens * 4,
        chunk_overlap=int((max_chunk_tokens * 4) / 10),
        add_start_index=True,
        strip_whitespace=True,
        separators=separators
    )
    print(f"- Fallback recursive splitter loaded\n")

    # Initialize tracking variables
    all_chunks = []
    stats = {"semantic_ok": 0, "fallback_used": 0, "total_chunks": 0, "skipped": 0}
    start_time = time.time()

    for doc_idx, document in enumerate(tqdm(documents, desc="- Processing documents")):
        doc_chunks = []
        used_semantic = False

        # Option 1: Try semantic chunking
        if semantic_chunker is not None:
            try:
                text = document.page_content.strip()
                raw_chunks = semantic_chunker.split_text(text)

                if not raw_chunks:
                    raise ValueError("No chunks generated by SemanticChunker")

                # Length check for raw chunks
                for chunk_idx, raw_chunk in enumerate(raw_chunks):
                    valid_chunks = split_if_too_big(
                        raw_chunk,
                        max_chunk_tokens,
                        min_chunk_tokens,
                        doc_idx,
                        chunk_idx
                    )
                    doc_chunks.extend(valid_chunks)

                # Apply overlapping logic
                if overlap_ratio > 0 and len(valid_chunks) > 1:
                    doc_chunks = create_overlap(
                        chunks=doc_chunks,
                        overlap_ratio=overlap_ratio,
                        max_chunk_tokens=max_chunk_tokens
                    )

                used_semantic = True
                stats["semantic_ok"] += 1

            except Exception as e:
                print(f"- SemanticChunker failed for doc {doc_idx}: {str(e)}")
                doc_chunks = []

        # Option 2: Use fallback splitter
        if not used_semantic:
            try:
                fallback_docs = fallback_splitter.split_documents([document])
                for fallback_doc in fallback_docs:
                    valid_chunks = split_if_too_big(
                        fallback_doc.page_content,
                        max_chunk_tokens,
                        min_chunk_tokens,
                        doc_idx,
                        0
                    )
                    doc_chunks.extend(valid_chunks)
                stats["fallback_used"] += 1

            except Exception as e:
                print(f"❌ Fallback splitter also failed for doc {doc_idx}: {str(e)}\n")
                stats["skipped"] += 1
                continue

        # Final step: convert chunks to LangchainDocument format with metadata
        for chunk_idx, chunk_content in enumerate(doc_chunks):
            final_token_count = count_tokens(chunk_content)

            # Duble check on the final token count
            if final_token_count > max_chunk_tokens:
                print(f"❌ ERROR: Chunk {doc_idx}_{chunk_idx} has {final_token_count} tokens!\n")
                continue

            if final_token_count < min_chunk_tokens:
                print(f"- Skipping chunk {doc_idx}_{chunk_idx}: too small ({final_token_count} tokens)\n")
                continue

            global EMBEDDING_MODEL_NAME
            chunk_doc = LangchainDocument(
                page_content=chunk_content,
                metadata={
                    "model name": EMBEDDING_MODEL_NAME,
                    "source_doc_idx": doc_idx,
                    "chunk_idx": chunk_idx,
                    "tokens": final_token_count,
                    "method": "semantic" if used_semantic else "fallback",
                    "max_chunk_tokens": max_chunk_tokens,
                    "min_chunk_tokens": min_chunk_tokens,
                    "similarity_threshold": similarity_threshold,
                    "overlap_ratio": overlap_ratio
                }
            )
            all_chunks.append(chunk_doc)
            stats["total_chunks"] += 1

    # Remove duplicate chunks
    unique_texts = {}
    uniq_all_chunks = []
    for doc in all_chunks:
        if doc.page_content not in unique_texts:
            unique_texts[doc.page_content] = True
            uniq_all_chunks.append(doc)

    # Clean up memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Report final statistics
    total_time = time.time() - start_time

    if uniq_all_chunks:
        token_counts = [chunk.metadata.get('tokens', 0) for chunk in uniq_all_chunks]
        avg_tokens = np.mean(token_counts)
        max_tokens_actual = max(token_counts)
        min_tokens_actual = min(token_counts)

        print("=" * 50)
        print(f"\n- Document Splitting Complete!\n")
        print("=" * 50)
        print(f"\n- FINAL REPORT:\n")
        print(f"- Documents processed: {len(documents)} → Chunks created: {len(uniq_all_chunks)}\n")
        print(f"- Total time: {total_time:.2f}s ({len(documents)/total_time:.1f} docs/sec)\n")
        print(f"- Chunk size (tokens): min={min_tokens_actual}, avg={avg_tokens:.1f}, max={max_tokens_actual}\n")
        print(f"- Methods used: {stats['semantic_ok']} semantic, {stats['fallback_used']} fallback, {stats['skipped']} skipped\n")

        # Plot token distribution
        plt.figure(figsize=(12, 8))
        sns.set(style="whitegrid")

        ax = sns.histplot(token_counts, bins=25, kde=True, color='dodgerblue', edgecolor='white')
        plt.title('Distribution of Token Counts per Chunk', fontsize=16, fontweight='bold')
        plt.xlabel('Number of Tokens', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.grid(True, linestyle='--', color='gray', alpha=0.15)
        sns.despine(left=True, bottom=True)
        plt.tight_layout()
        plt.show()

    return uniq_all_chunks


### **EMBEDDING MODEL** 

We now load the best embedding model identified during the evaluation phase: **`pritamdeka/S-PubMedBert-MS-MARCO`**, which showed the highest retrieval performance on our test ($\sim 67 \%$).

This model will be used to encode the documents and questions throughout the pipeline.

In [ ]:
# Embedding Model
EMBEDDING_MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO" #<-- The best one found so far

# Tokenizer
TOKENIZER = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

embedding_model = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            model_kwargs={"device": 'cuda'},
            encode_kwargs={
                "batch_size": 64,
                "normalize_embeddings": True
            }
        )

### **DOC SPLITTING**

We load the raw corpus and convert each entry into a `LangchainDocument`.  
Then, we apply semantic chunking using the selected embedding model to generate smaller self-contained passages.  
Chunks are saved in a JSON file and reloaded later for retrieval.

This ensures that each chunk is optimized for dense retrieval within the RAG framework.

In [ ]:
# ======================================
# Load and Prepare Corpus
# ======================================

# Load the corpus CSV with explicit separator
corpus_df = pd.read_csv("/kaggle/input/hackathon-data/corpus.csv", sep=",")

# Display the available column names
print("CSV columns:", corpus_df.columns)

# Create a working copy of the corpus
corpus_df_sample = corpus_df.copy()

# Ensure proper data types
corpus_df_sample["id"] = corpus_df_sample["id"].astype(int)
corpus_df_sample["passage"] = corpus_df_sample["passage"].astype(str)

# Convert corpus rows into LangchainDocument objects
corpus_documents = [
    LangchainDocument(
        page_content=row["passage"],
        metadata={"id": row["id"]}
    )
    for _, row in corpus_df_sample.iterrows()
]

print("- LangChain Documents created:", len(corpus_documents))

In [ ]:
# ======================================
# Chunking the Documents
# ======================================

# Create semantic chunks from Langchain documents using an embedding model
knowledge_chunks = create_doc_chunks(
    documents=corpus_documents,
    model=embedding_model,             # Embedding model used to compute similarity
    max_chunk_tokens=500,              # Maximum tokens per chunk (hard limit)
    min_chunk_tokens=100,              # Minimum threshold to ensure quality
    similarity_threshold=85,           # Empirically chosen similarity threshold
    device="cuda",                     # Force GPU usage if available
    batch_size=64,                     # Optimized batch size for performance (e.g., Kaggle GPUs)
    overlap_ratio=0.10                 # Overlap between adjacent chunks to preserve context
)


In [ ]:
# ======================================
# Save Chunked Knowledge to JSON
# ======================================

# Convert LangchainDocument objects into plain dictionaries
chunk_dicts = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in knowledge_chunks
]

# Save the chunked data to a JSON file
output_path = f"{500}_biomed_knw_chunks.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunk_dicts, f, ensure_ascii=False, indent=2)

print(f"- Saved knowledge_chunks to {output_path}")

In [ ]:
# ======================================
# Reload Preprocessed Chunks from JSON
# ======================================

# Load the saved JSON file containing pre-chunked documents
with open("/kaggle/input/d1-models-file/500_biomed_knw_chunks.json", "r", encoding="utf-8") as f:
    chunk_dicts = json.load(f)

# Convert dictionaries back to LangchainDocument objects
knowledge_chunks = [
    LangchainDocument(
        page_content=chunk["page_content"],
        metadata=chunk["metadata"]
    )
    for chunk in chunk_dicts
]

print(f"- Loaded {len(knowledge_chunks)} chunks from JSON file")

### **VECTOR STORE CREATION**

We build a FAISS vector store using the chunked documents and the selected embedding model.  
Cosine similarity is used as the distance metric.  
The vector store is saved locally for future retrieval operations.

In [ ]:
knw_vector_db = FAISS.from_documents(
    documents=knowledge_chunks,
    embedding=embedding_model,
    distance_strategy=DistanceStrategy.COSINE
)

knw_vector_db.save_local("ms_marco_faiss_index_dir")

### **VECTOR STORE LOADING**

We load the previously saved FAISS vector index from disk, using the same embedding model.  
This allows us to resume retrieval without recomputing document embeddings.

In [ ]:
knw_vector_db = FAISS.load_local("ms_marco_faiss_index_dir", 
                                 embedding_model, 
                                 allow_dangerous_deserialization=True, 
                                 distance_strategy=DistanceStrategy.COSINE,)

### **READER AND RERANKER SETUP**

In this section, we configure the two main components of the RAG pipeline:

- **Reranker**: We use `colbert-ir/colbertv2.0` to re-score retrieved passages based on their relevance to the query.
- **Reader**: A quantized version of `meta-llama/Llama-3.1-8B-Instruct` is loaded with 4-bit precision to generate answers efficiently.
  
We also define a structured prompt template tailored for biomedical question answering.

In [ ]:
# ======================================
# Load Reranker Model (ColBERTv2)
# ======================================

# ColBERTv2 reranker for re-scoring retrieved passages
RERANKER = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

In [ ]:
# ===========================================================
# Quantization Configuration (4-bit) for Reader Model
# ===========================================================

from transformers import BitsAndBytesConfig

# Configure 4-bit quantization for memory-efficient inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
# ================================================================================
# Load the meta-llama Llama-3.1-8B-Instruct model with 4-bit quantization
# ================================================================================
login("hf**********") # Replace with your Hugging Face token

READER_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    READER_MODEL_NAME, 
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

reader_tokenizer = AutoTokenizer.from_pretrained(READER_MODEL_NAME)

In [ ]:
# ======================================
# Tokenizer Configuration for Generation
# ======================================

# Ensure padding token is defined for batch generation
if reader_tokenizer.pad_token is None:
    reader_tokenizer.pad_token = reader_tokenizer.eos_token
    reader_tokenizer.pad_token_id = reader_tokenizer.eos_token_id

# Set tokenizer behavior for generation
reader_tokenizer.padding_side = "left"
reader_tokenizer.truncation_side = "left"

In [ ]:
# ======================================
# Create Reader Pipeline
# ======================================

from transformers import pipeline

READER_LLM = pipeline(
    model=model,
    tokenizer=reader_tokenizer,
    task="text-generation",
    do_sample=True,            # Enable sampling (set False for deterministic)
    temperature=0.2,           # Low temperature for more focused outputs
    repetition_penalty=1.1,    # Discourage repetition
    return_full_text=False,    # Return only the generated answer
    max_new_tokens=300         # Limit output length
)

In [ ]:
# ==============================================
# Chat Prompt Template for Biomedical RAG
# ==============================================

prompt_in_chat_format = [
    {
        "role": "system",
        "content": """You are a biomedical expert. Provide concise, precise, and detailed scientific answers based strictly on the provided context.
        ⁠If the question can be answered directly with "Yes" or "No", always start your response explicitly with "Yes." or "No.", followed by a brief scientific justification.
        ⁠For all other types of questions, provide a direct scientific answer, explicitly mentioning relevant molecules, algorithms, conditions, or treatments.
        ⁠Do NOT use phrases like "according to the context" or "according to document X". If the information needed to precisely answer the question is missing in the provided context, reply exactly: "The answer is not available in the context." """
    },
    {
        "role": "user",
        "content": """Context:
{context}
---
Question: {question}"""
    }
]

RAG_PROMPT_TEMPLATE = reader_tokenizer.apply_chat_template(
    prompt_in_chat_format,
    tokenize=False,
    add_generation_prompt=True
)

### **RAG PIPELINE**

Finally, we set up the RAG pipeline to cobine all the phases.

We now define the complete Retrieval-Augmented Generation (RAG) function.  
This function combines retrieval from the FAISS index, reranking with ColBERT, and answer generation with LLaMA 3.  
The top-k retrieved documents are reranked, and the most relevant ones are used as context in the prompt passed to the reader model.

In [ ]:
# ===========================================================
# RAG Function with Retrieval, Reranking, and Generation
# ===========================================================

def rag(
    query: str,
    llm: Pipeline,
    knowledge_index: FAISS,
    reranker: RAGPretrainedModel,
    num_retrived_doc: int = 20,
    num_context_doc: int = 5,
) -> Tuple[str, List[LangchainDocument]]:
    """
    End-to-end Retrieval-Augmented Generation (RAG) pipeline.
    
    Steps:
    1. Retrieve top-k documents from FAISS.
    2. Rerank documents using a ColBERT reranker.
    3. Construct a prompt with top reranked documents.
    4. Generate an answer using the language model.
    
    Args:
        query (str): User question.
        llm (Pipeline): HuggingFace pipeline for text generation.
        knowledge_index (FAISS): FAISS vector DB retriever.
        reranker (RAGPretrainedModel): ColBERT reranker.
        num_retrived_doc (int): Number of documents to retrieve initially.
        num_context_doc (int): Number of top reranked documents to include in prompt.
    
    Returns:
        Tuple[str, List[LangchainDocument]]: Generated answer and original retrieved documents.
    """

    # Step 1: Retrieve candidate documents using FAISS
    retrieved_docs = knowledge_index.similarity_search(query=query, k=num_retrived_doc)
    retrieved_doc_texts = [doc.page_content for doc in retrieved_docs]

    # Step 2: Rerank the retrieved documents with ColBERT
    reranked = reranker.rerank(query, retrieved_doc_texts, k=num_context_doc)
    reranked_docs = [doc["content"] for doc in reranked]

    # Step 3: Format context for the prompt
    context = "\nExtracted documents:\n"
    context += "".join(
        [f"Document {i}:::\n{doc}\n" for i, doc in enumerate(reranked_docs)]
    )

    # Step 4: Create final prompt and generate the answer
    final_prompt = RAG_PROMPT_TEMPLATE.format(question=query, context=context)
    response = llm(final_prompt)[0]["generated_text"]

    return response


## **RAG EXECUTION**

We run a test query through the full RAG pipeline to validate that retrieval, reranking, and generation are working correctly.

In [ ]:
rag("Name the algorithms for counting multi-mapping reads", READER_LLM, knw_vector_db, RERANKER, 30, 5)

### **SUBMISSION FILE CREATION**

In this final step, we generate the submission file for evaluation.  
Each question in the test set is processed using the full RAG pipeline (retrieval, reranking, and generation), and the predicted answers are collected.  
The results are saved in a CSV file in the format required for submission.

In [ ]:
# ======================================
# Submission File Generation
# ======================================
# Load the test set (in this case, using the training file as a stand-in)
test = pd.read_csv("/kaggle/input/adm-lt-2024-2025-hackathon-rag/train.csv")

# Initialize the submission dictionary to store question IDs and generated answers
submission = {"id": [], "answer": []}

# Loop through each question in the test set
for idx, row in tqdm(test.iterrows()):
    query_id = row["id"]                      # Extract the question ID
    query_text = row["question"]              # Extract the question text
    
    # Generate the answer using the RAG pipeline
    answer = rag(query_text, READER_LLM, knw_vector_db, RERANKER, 30, 5)
    
    # Store the results in the submission dictionary
    submission["id"].append(query_id)
    submission["answer"].append(answer)

# Convert the submission dictionary to a DataFrame
submission_df = pd.DataFrame(submission)

# Save the submission to a CSV file (Kaggle-style output format)
submission_df.to_csv("submission.csv", index=False)

## **End of Pipeline**

The entire RAG workflow has been successfully implemented and tested.  

Documents were embedded and chunked, the vector index was created, retrieval and reranking were applied, and answers were generated and saved for submission.